### If you get an error here, it means you need to run the prepare_data.ipynb file first

In [1]:
import json
import numpy as np
import pandas as pd

from rnanorm import TMM
from pathlib import Path


description_path = Path('description.json')
if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    data = json.load(f)

## Preprocessing steps

1. **Load data** — Read the dataset and split columns into clinical variables and raw gene-level counts.

2. **Robust-expression filter (defined on raw counts)** — Keep genes with the 75th percentile of raw counts across samples \(\ge 100\) (i.e., expressed at \(\ge 100\) counts in at least 25% of samples).

3. **TMM normalization** — Apply TMM to the full gene count matrix to compute normalization factors correcting for library size and RNA composition; obtain normalized counts.

4. **Apply gene filter and log-transform** — Subset to the robustly expressed genes and apply \(\log(x+1)\).

5. <strike>**Z-score standardization** — For each gene, center and scale across samples to mean 0 and standard deviation 1.</strike>

In [2]:
data_file_name = data['data_file_names'][0]

num_clinical_features = data['data_summary']['clinical_features']

df = pd.read_csv(data_file_name)


clinical_cols = df.columns[:num_clinical_features]
gene_cols = df.columns[num_clinical_features:]

raw_counts = df[gene_cols]
print('Before preprocessing:\n - n_genes: %d' % (df[gene_cols].shape[1]))

# create mask of robustly expressed genes
robus_genes_mask = np.percentile(df[gene_cols], 75, axis=0) >= 100

# normalize counts
df[gene_cols] = TMM().set_output(transform="pandas").fit_transform(df[gene_cols])

# apply robust gene mask
gene_cols = df[gene_cols].columns[robus_genes_mask].tolist()
df = df[list(clinical_cols) + gene_cols]

# # log transform and standardize (genes only)
df[gene_cols] = np.log(df[gene_cols] + 1)
# df[gene_cols] = (df[gene_cols] - df[gene_cols].mean(axis=0)) / df[gene_cols].std(axis=0)

print("\nAfter preprocessing:\n - n_genes: %d" % (df[gene_cols].shape[1]))

transformed_data_file_name = data_file_name.replace("_raw_counts_NCBI", "_preprocessed")


/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_13799/2835454030.py:5: DtypeWarning: Columns (7,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_file_name)


Before preprocessing:
 - n_genes: 39376



After preprocessing:
 - n_genes: 16086


In [3]:
url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
df_gene_annotations = df_gene_annotations[df_gene_annotations["Status"] == "active"]
df_gene_annotations = df_gene_annotations[df_gene_annotations["GeneType"] == "protein-coding"]
df_gene_annotations = df_gene_annotations.reset_index(drop=True)

protein_coding_genes = set(df_gene_annotations["Symbol"].dropna())
gene_cols = [gene for gene in gene_cols if gene in protein_coding_genes]
df = df[list(clinical_cols) + gene_cols]

print("\nAfter protein-coding filter:\n - n_genes: %d" % len(gene_cols))

df.to_csv(transformed_data_file_name, index=False)

if transformed_data_file_name not in data['data_file_names']:
    data['data_file_names'].append(transformed_data_file_name)

with open(description_path, 'w') as f:
    json.dump(data, f, indent=4)

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_13799/4029965179.py:2: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(



After protein-coding filter:
 - n_genes: 13979


## Survival variables encodings

Authors encode survival variables in various ways, here we have just a check if all the survival variables are already numeric. If they are not, we convert them by hand.


In [4]:
df_ = df.copy()

endpoints = []
for endpoint in data["survival-endpoints"]:
    endpoint_time = endpoint["time_var"]["var_name"]
    endpoint_event = endpoint["event_var"]["var_name"]

    if endpoint_time == "unknown" or endpoint_event == "unknown":
        continue

    # Replace "Not available" and similar missing-value strings with NaN
    for col in [endpoint_time, endpoint_event]:
        df_[col] = df_[col].replace(["Not available", "NA", "N/A", "not available", "-"], pd.NA)


    # make sure time and event are numeric
    df_[endpoint_time] = pd.to_numeric(df_[endpoint_time])
    df_[endpoint_event] = df_[endpoint_event].map({"Dead": 1, "Y": 1, "Alive": 0, "N": 0})
    df_[endpoint_event] = pd.to_numeric(df_[endpoint_event])
    
    endpoints.append((endpoint_time, endpoint_event))


# Flatten endpoints and use it to order survival columns
surv_cols = [col for tup in endpoints for col in tup]
other_cols = [col for col in df_.columns if col not in surv_cols]
ordered_cols = surv_cols + other_cols

df_[ordered_cols].to_csv(transformed_data_file_name, index=False)

In [5]:
df_[ordered_cols]

,pfs.mos..,recurrence,os.mos..,vitalstatus,sample_title,stic,age,path.stage,surgical.outcome,diagnosis.of.record,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,19.0,1,34.0,1.0,HGSC_40190,Yes,61,III,Optimal,Not available,...,6.040067,5.035451,5.740524,6.091217,4.829373,5.089559,6.619170,6.902299,6.207234,6.313062
1,40.0,1,64.0,0.0,HGSC_40191,No,49,III,Optimal,Not available,...,6.537236,5.038692,5.921121,6.717286,5.075715,5.334657,6.859956,6.920267,6.185481,6.279690
2,42.0,1,72.0,0.0,HGSC_40193,No,53,II,Unknown,Not available,...,7.070010,6.296203,7.057889,6.890579,5.866339,6.279101,7.889931,8.161322,7.570871,7.458223
3,32.0,1,34.0,1.0,HGSC_40197,Yes,53,III,Optimal,Not available,...,6.976777,6.125367,6.876939,7.116488,5.835287,6.258743,7.824677,8.059266,7.367960,7.466249
4,13.0,1,39.0,1.0,HGSC_40199,No,49,III,Suboptimal,Not available,...,6.490908,5.032042,5.987931,6.960127,5.560689,5.339411,6.856786,7.024983,6.217915,6.551331
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,45.5,0,45.5,0.0,HGSC_JH1072,Yes,54,IV,Optimal,Ovarian Ca,...,6.412548,3.447208,5.442689,6.977427,5.223792,5.420394,6.705600,6.593751,4.957902,6.338230
80,24.7,1,63.6,0.0,HGSC_JH1075,Yes,43,IV,Suboptimal,Ovarian Ca,...,6.527918,3.383243,5.655455,7.365742,4.767843,4.806073,6.239238,6.011015,3.772743,6.092342
81,54.7,0,54.7,0.0,HGSC_JH1085,No,63,IA,Optimal,Ovarian Ca,...,7.263524,3.554087,6.031036,8.343936,5.315515,5.106674,6.675686,6.350713,4.241415,6.359426
82,74.6,0,74.6,0.0,HGSC_JH993,No,73,IIIA,Optimal,Ovarian Ca,...,6.287893,4.918423,5.734574,6.897196,4.916261,4.905930,6.567011,6.615294,5.921755,6.283216
